In [10]:
import os
import pandas as pd

# Define paths
root_path = "../package_metadata/openml"
target_folder = "kernels_iid_comparison"

# Define the coefficients list corresponding to sorted target sizes
coeffs_template = ["1/4", "1/2", "1", "2", "4"]

dataframes_list = []

# Check if root path exists
if os.path.exists(root_path):
    # Iterate through all dataset folders
    for dataset_name in os.listdir(root_path):
        dataset_full_path = os.path.join(root_path, dataset_name)
        
        # Ensure it is a directory
        if os.path.isdir(dataset_full_path):
            
            # Construct the path to the target kernels folder
            target_path = os.path.join(dataset_full_path, target_folder)
            
            # Check if the target folder exists in this dataset
            if os.path.exists(target_path):
                # Iterate through all files inside
                for filename in os.listdir(target_path):
                    if filename.endswith(".csv"):
                        file_path = os.path.join(target_path, filename)
                        
                        try:
                            # Read the CSV file
                            df = pd.read_csv(file_path)
                            
                            # --- 1. EXTRACT MODEL NAME FROM FILENAME ---
                            # Check specific substrings in the filename
                            if "_ann" in filename:
                                df["model_name"] = "nn"
                            elif "_xgboost" in filename:
                                df["model_name"] = "xgboost"
                            else:
                                # Fallback if neither is found (optional)
                                df["model_name"] = "unknown"
                            
                            # --- 2. EXISTING COLUMNS LOGIC ---
                            # If 'data_modification_method' is missing, add "none"
                            if "data_modification_method" not in df.columns:
                                df["data_modification_method"] = "none"
                            
                            # Add 'dataset_name' column
                            df["dataset_name"] = dataset_name
                            
                            # --- 3. COMPRESSION COEFF LOGIC ---
                            if "target_size" in df.columns:
                                # Get unique values and sort them (smallest to largest)
                                unique_sizes = sorted(df["target_size"].unique())
                                
                                # Check if the number of unique sizes matches our template (5 items)
                                if len(unique_sizes) == len(coeffs_template):
                                    # Create a dictionary mapping: {size_value: "coeff_string"}
                                    mapping = dict(zip(unique_sizes, coeffs_template))
                                    
                                    # Apply the mapping to create the new column
                                    df["compression_coeff"] = df["target_size"].map(mapping)
                                else:
                                    # Handle cases where unique sizes are not exactly 5
                                    raise ValueError(f"Unexpected number of unique target sizes in {filename}: found {len(unique_sizes)}, expected {len(coeffs_template)}")
                            else:
                                print(f"Warning: 'target_size' column missing in {filename}")
                            
                            dataframes_list.append(df)
                            
                        except Exception as e:
                            print(f"Error reading file {file_path}: {e}")

df_results = pd.concat(dataframes_list, ignore_index=True)

In [11]:
group_cols = [
    "target_size", 
    "model_name",
    "explainer", 
    "strategy", 
    "method", 
    "kernel", 
    "data_modification_method", 
    "dataset_name", 
    "compression_coeff"
]

# metrics to calculate
metrics_to_aggregate = {
    "mae": ["mean", "std"],
    "top_k": ["mean", "std"],             
    "compression_time": ["mean", "std"],
    "explanation_time": ["mean", "std"]
}

existing_group_cols = [col for col in group_cols if col in df_results.columns]

if len(existing_group_cols) == len(group_cols):
    # grouping
    grouped_df = df_results.groupby(existing_group_cols).agg(metrics_to_aggregate)
    new_columns = []
    for metric, stat in grouped_df.columns:
        new_columns.append(f"{stat}_{metric}")
    
    grouped_df.columns = new_columns
    
else:
    missing = set(group_cols) - set(df_results.columns)
    print(f"Error: The following grouping columns are missing in the DataFrame: {missing}")

In [12]:
grouped_df = grouped_df.reset_index()

In [13]:

def get_method_total(row):
    if row['method'] == 'iid':
        return 'iid'
    elif row['method'] == 'kernel_thinning':
        return f"kt_{row['kernel']}"

grouped_df['method_total'] = grouped_df.apply(get_method_total, axis=1)

In [14]:
def get_explainer_total(row):
    ex = row['explainer']
    st = row['strategy']
    
    if ex == 'expected_gradients':
        return 'expected_gradients'

    elif ex == 'shap' and st == 'kernel':
        return 'shap_kernel'
    
    elif ex == 'shapiq' and st == 'kernel':
        return 'shapiq_kernel'
        
    elif ex == 'sage' and st == 'permutation':
        return 'sage_permutation'

grouped_df['explainer_total'] = grouped_df.apply(get_explainer_total, axis=1)

In [15]:
cols_to_drop = [
    "target_size", 
    "explainer", 
    "strategy", 
    "method", 
    "kernel", 
    "data_modification_method", 
]

grouped_df = grouped_df.drop(columns=cols_to_drop, errors='ignore')
grouped_df

,model_name,dataset_name,compression_coeff,mean_mae,std_mae,mean_top_k,std_top_k,mean_compression_time,std_compression_time,mean_explanation_time,std_explanation_time,method_total,explainer_total
0,nn,GesturePhaseSegmentationProcessed,1/4,0.075259,0.014650,0.700373,0.044693,0.000229,0.000072,178.181576,1.746770,iid,expected_gradients
1,nn,first-order-theorem-proving,1/4,0.062936,0.009793,0.671304,0.062175,0.000199,0.000022,91.071833,1.575469,iid,expected_gradients
2,nn,har,1/4,0.013010,0.001874,0.738408,0.010741,0.005915,0.006129,291.640962,1.068811,iid,expected_gradients
3,nn,isolet,1/4,0.014647,0.001689,0.749979,0.026535,0.000353,0.000054,222.196147,6.113391,iid,expected_gradients
4,nn,optdigits,1/4,0.110294,0.023303,0.760157,0.067148,0.000302,0.000343,84.842777,1.302012,iid,expected_gradients
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4645,xgboost,video_transcoding,4,0.003624,0.000220,0.966675,0.003001,5.036373,0.036100,17732.643599,366.899726,kt_inverse_multiquadric,shap_kernel
4646,xgboost,numerai28.6,4,0.001926,0.000044,0.907949,0.002589,6.692274,0.058874,18964.163453,450.681185,kt_matern,shap_kernel
4647,xgboost,video_transcoding,4,0.003634,0.000285,0.968379,0.003309,6.491528,0.034393,17728.651109,390.327200,kt_matern,shap_kernel
4648,xgboost,numerai28.6,4,0.038840,0.000751,0.306855,0.000815,15.337061,0.103742,18913.498517,345.801391,kt_sobolev,shap_kernel


# Image 1

In [7]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

coeff_order = ["1/4", "1/2", "1", "2", "4"]

metrics = {
    "mean_mae": "Mean MAE",
    "mean_top_k": "Mean Top-K",
    "mean_compression_time": "Mean Compression Time", 
    "std_mae": "STD MAE",
    "std_top_k": "STD Top-K",
    "std_compression_time": "STD Compression Time"
}

def plot_metric_image1(metric_name, metric_label):
    output_folder = f"Image1/{metric_name}"
    os.makedirs(output_folder, exist_ok=True)

    explainers = grouped_df['explainer_total'].unique()

    for explainer in explainers:
        fig, axes = plt.subplots(1, 2, figsize=(20, 8), sharey=True)
        fig.suptitle(f"{metric_label}: {explainer}", fontsize=18)

        for ax, model in zip(axes, ["nn", "xgboost"]):
            subset = grouped_df[
                (grouped_df['model_name'] == model) &
                (grouped_df['explainer_total'] == explainer) &
                (grouped_df['method_total'] != "kt_sobolev")
            ]

            if subset.empty:
                ax.set_title(f"{model} (no data)")
                ax.axis("off")
                continue

            sns.boxplot(
                data=subset,
                x="compression_coeff",
                y=metric_name,
                hue="method_total",
                order=coeff_order,
                palette="YlOrBr",
                linewidth=1.2,
                fliersize=4,
                ax=ax
            )

            if metric_name in ["mean_mae", "mean_compression_time", "std_mae", "std_compression_time"]:
                ax.set_yscale("log")

            ax.set_title(f"{model}", fontsize=15)
            ax.set_xlabel("Compression Coefficient")
            ax.set_ylabel(metric_label if model == "nn" else "")

            ax.grid(axis='y', linestyle='--', alpha=0.7)

        handles, labels = axes[0].get_legend_handles_labels()

        plt.tight_layout()

        filename = f"{output_folder}/{metric_name}_{explainer}.png"
        plt.savefig(filename, bbox_inches='tight')
        plt.close()


for metric_name, metric_label in metrics.items():
    plot_metric_image1(metric_name, metric_label)


# Image 2

In [17]:
grouped_df = grouped_df[grouped_df["method_total"] != "kt_sobolev"]
grouped_df = grouped_df[grouped_df["method_total"] != "iid"]

In [18]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from matplotlib.colors import LinearSegmentedColormap

df = grouped_df.copy()

df['method_total'] = df['method_total'].replace({
    'kt_matern': 'Matern',
    'kt_inverse_multiquadric': 'Inverse-Multi',
    'kt_gaussian': 'Gaussian'
})

metrics = {
    "mean_mae": "Mean MAE",
    "mean_top_k": "Mean Top-K",
    "mean_compression_time": "Mean Compression Time",
    "std_mae": "STD MAE",
    "std_top_k": "STD Top-K",
    "std_compression_time": "STD Compression Time"
}

higher_is_better = ["mean_top_k"]

methods = df['method_total'].unique()
explainers = df['explainer_total'].unique()

heatmap_cmap = LinearSegmentedColormap.from_list("custom_heatmap", ['#FFDFB9', '#A4193D'])
barplot_palette = ['#A4193D', '#FFDFB9']

for metric, metric_label in metrics.items():
    output_folder = f"Image2/{metric}"
    os.makedirs(output_folder, exist_ok=True)

    for explainer in explainers:
        fig, axes = plt.subplots(1, 3, figsize=(28, 10)) 
        fig.suptitle(f"Head-to-head comparison ({metric_label}): {explainer}", fontsize=40)

        total_wins_dict = {}

        for idx, model in enumerate(["nn", "xgboost"]):
            ax_heat = axes[idx*2] 

            comparison_matrix = pd.DataFrame(0, index=methods, columns=methods)
            subset_main = df[(df['model_name'] == model) & (df['explainer_total'] == explainer)]

            for dataset in subset_main['dataset_name'].unique():
                for coeff in subset_main['compression_coeff'].unique():
                    subset = subset_main[
                        (subset_main['dataset_name'] == dataset) &
                        (subset_main['compression_coeff'] == coeff)
                    ]

                    for method_i in methods:
                        for method_j in methods:
                            if method_i == method_j:
                                continue
                            values_i = subset[subset['method_total'] == method_i][metric].values
                            values_j = subset[subset['method_total'] == method_j][metric].values
                            if len(values_i) == 0 or len(values_j) == 0:
                                continue
                            if metric in higher_is_better:
                                count = np.sum(values_i[:, None] > values_j)
                            else:
                                count = np.sum(values_i[:, None] < values_j)
                            comparison_matrix.loc[method_i, method_j] += count

            total_wins_dict[model] = comparison_matrix.sum(axis=1)

            sns.heatmap(
                comparison_matrix,
                annot=True,
                fmt="d",
                cmap=heatmap_cmap,
                annot_kws={"size": 35},
                cbar=False, 
                ax=ax_heat
            )
            ax_heat.set_title(f"Model: {model}", fontsize=25)
            
            ax_heat.set_xlabel(r"Method $j$", fontsize=25)
            ax_heat.set_ylabel(r"Method $i$", fontsize=25)
            
            ax_heat.tick_params(axis='x', labelsize=20)
            ax_heat.tick_params(axis='y', labelsize=20)

        ax_bar = axes[1]
        bar_data = pd.DataFrame({
            'method': list(methods)*2,
            'model': ['nn']*len(methods) + ['xgboost']*len(methods),
            'total_wins': list(total_wins_dict['nn'].values) + list(total_wins_dict['xgboost'].values)
        })
        sns.barplot(
            data=bar_data,
            x='method',
            y='total_wins',
            hue='model',
            palette=barplot_palette,
            ax=ax_bar
        )
        ax_bar.set_title("Total wins per method", fontsize=25)
        
        ax_bar.set_xlabel("", fontsize=25)
        ax_bar.set_ylabel("", fontsize=25)
        
        ax_bar.tick_params(axis='x', labelsize=25)
        ax_bar.tick_params(axis='y', labelsize=25)
        
        ax_bar.legend(title="Model", fontsize=20, title_fontsize=25)

        plt.tight_layout(rect=[0, 0, 0.95, 0.95])
        filename = f"{output_folder}/head_to_head_grouped_bar_{explainer}.png"
        plt.savefig(filename, bbox_inches='tight')
        plt.close()

# Image 4

In [22]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

df = grouped_df.copy()

metrics = {
    "mean_mae": "Mean MAE",
    "mean_top_k": "Mean Top-K",
    "mean_compression_time": "Mean Compression Time",
}

higher_is_better = ["mean_top_k"]

methods = sorted(df['method_total'].unique())
explainers = df['explainer_total'].unique()
models = ["nn", "xgboost"]

palette = sns.color_palette("tab10", n_colors=len(methods))
method_colors = dict(zip(methods, palette))

def get_coeff_label(val):
    try:
        f = float(val)
        if np.isclose(f, 0.25): return "1/4"
        if np.isclose(f, 0.5): return "1/2"
        if np.isclose(f, 1.0): return "1"
        if np.isclose(f, 2.0): return "2"
        if np.isclose(f, 4.0): return "4"
    except (ValueError, TypeError):
        pass
    return str(val)

target_order = ["1/4", "1/2", "1", "2", "4"]

for metric, metric_label in metrics.items():
    output_folder = f"Image4/{metric}"
    os.makedirs(output_folder, exist_ok=True)

    is_higher_better = metric in higher_is_better

    for explainer in explainers:
        fig, axes = plt.subplots(1, 2, figsize=(20, 8))
        fig.suptitle(f"Win Trend vs Compression Coeff ({metric_label}): {explainer}", fontsize=24)

        for idx, model in enumerate(models):
            ax = axes[idx]

            subset_main = df[
                (df['model_name'] == model) &
                (df['explainer_total'] == explainer)
            ]

            wins_data = []

            unique_coeffs = sorted(subset_main['compression_coeff'].unique())

            for coeff in unique_coeffs:
                subset_coeff = subset_main[subset_main['compression_coeff'] == coeff]

                if subset_coeff.empty:
                    continue

                pivot_df = subset_coeff.pivot(index='dataset_name', columns='method_total', values=metric)

                for method in methods:
                    if method not in pivot_df.columns:
                        wins_data.append({
                            'compression_coeff': coeff,
                            'method_total': method,
                            'wins': 0
                        })
                        continue

                    current_method_vals = pivot_df[method]

                    total_wins_at_this_coeff = 0

                    for competitor in methods:
                        if method == competitor or competitor not in pivot_df.columns:
                            continue

                        competitor_vals = pivot_df[competitor]

                        if is_higher_better:
                            wins = (current_method_vals > competitor_vals).sum()
                        else:
                            wins = (current_method_vals < competitor_vals).sum()

                        total_wins_at_this_coeff += wins

                    wins_data.append({
                        'compression_coeff': coeff,
                        'method_total': method,
                        'wins': total_wins_at_this_coeff
                    })

            plot_df = pd.DataFrame(wins_data)

            if not plot_df.empty:
                plot_df['coeff_str'] = plot_df['compression_coeff'].apply(get_coeff_label)

                present_labels = plot_df['coeff_str'].unique()
                extras = [l for l in present_labels if l not in target_order]
                final_order = target_order + sorted(extras)

                plot_df['coeff_str'] = pd.Categorical(
                    plot_df['coeff_str'],
                    categories=final_order,
                    ordered=True
                )

                sns.lineplot(
                    data=plot_df,
                    x='coeff_str',
                    y='wins',
                    hue='method_total',
                    palette=method_colors,
                    marker='o',
                    linewidth=2.5,
                    ax=ax,
                    sort=True
                )

                ax.set_title(f"Model: {model}", fontsize=18)
                ax.set_xlabel("Compression Coefficient", fontsize=14)
                ax.set_ylabel("Total Pairwise Wins", fontsize=14)
                ax.grid(True, linestyle='--', alpha=0.5)

                if idx == 0:
                    ax.get_legend().remove()
                else:
                    ax.legend(title='Method', bbox_to_anchor=(1.05, 1), loc='upper left')

        plt.tight_layout()
        filename = f"{output_folder}/wins_trend_{explainer}.png"
        plt.savefig(filename, bbox_inches='tight')
        plt.close()

print("All trend plots generated.")

All trend plots generated.
